# 02 — Matrices diarias OD EODH

Construye productos diarios dispersos (formato largo) desde la SQLite relacional existente. Cuenta todos los viajes con códigos de origen y destino no vacíos; conserva literalmente los códigos, los pares no clasificados y los intrazonales. La expansión técnica aplica `fexp_hg` por viaje, de acuerdo con la fórmula documentada de Etapa IV; `fexp_vj` se conserva como contraste. No hay filtros por modo, propósito, período ni territorio.

Entradas: `data/relational/em2023_relational.sqlite` (`viajes`, `hogares`). Salidas: cuatro archivos CSV/Parquet en `outputs/matrices/`, tres tablas QA en `outputs/tables/` y el método en `outputs/reports/`.

In [ ]:
from pathlib import Path
from decimal import Decimal
import sqlite3
import pandas as pd
from mode_choice.od_matrix import build_od_products

PROJECT_ROOT = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / 'pyproject.toml').is_file()), None)
assert PROJECT_ROOT is not None, 'Could not locate the project root.'
result = build_od_products(PROJECT_ROOT)
result['summary']

In [ ]:
sample = pd.read_parquet(PROJECT_ROOT / 'outputs/matrices/od_daily_sample.parquet')
expanded = pd.read_parquet(PROJECT_ROOT / 'outputs/matrices/od_daily_expanded.parquet')
sample_csv = pd.read_csv(PROJECT_ROOT / 'outputs/matrices/od_daily_sample.csv', dtype=str, keep_default_na=False)
expanded_csv = pd.read_csv(PROJECT_ROOT / 'outputs/matrices/od_daily_expanded.csv', dtype=str, keep_default_na=False)
reconciliation = pd.read_csv(PROJECT_ROOT / 'outputs/tables/od_reconciliation.csv', dtype=str)
quality = pd.read_csv(PROJECT_ROOT / 'outputs/tables/od_data_quality.csv', dtype=str)

assert int(sample['sample_trips'].sum()) == 100174
assert int(reconciliation.loc[reconciliation.metric == 'intrazonal_sample_trips', 'value'].iloc[0]) == 15016
assert int(reconciliation.loc[reconciliation.metric == 'trips_included_beyond_legacy_study_area_scope', 'value'].iloc[0]) == 433
assert sample[['zat_ori', 'zat_des']].equals(expanded[['zat_ori', 'zat_des']])
assert sample[['zat_ori', 'zat_des']].equals(sample_csv[['zat_ori', 'zat_des']])
assert expanded[['zat_ori', 'zat_des']].equals(expanded_csv[['zat_ori', 'zat_des']])
assert sample['sample_trips'].tolist() == sample_csv['sample_trips'].astype('int64').tolist()
assert expanded['expanded_trips'].tolist() == [Decimal(value) for value in expanded_csv['expanded_trips']]
assert 'key_viaje' not in sample.columns and 'key_hg' not in sample.columns
assert 'key_viaje' not in expanded.columns and 'key_hg' not in expanded.columns

database_path = PROJECT_ROOT / 'data/relational/em2023_relational.sqlite'
connection = sqlite3.connect(f'file:{database_path.as_posix()}?mode=ro', uri=True)
try:
    source_pairs = connection.execute('''SELECT "zat_ori", "zat_des", COUNT(*)
        FROM "viajes"
        WHERE "zat_ori" IS NOT NULL AND length(trim("zat_ori")) > 0
          AND "zat_des" IS NOT NULL AND length(trim("zat_des")) > 0
        GROUP BY "zat_ori", "zat_des"''').fetchall()
finally:
    connection.close()
source_pair_counts = {(origin, destination): count for origin, destination, count in source_pairs}
matrix_pair_counts = {(row.zat_ori, row.zat_des): int(row.sample_trips) for row in sample.itertuples(index=False)}
assert source_pair_counts == matrix_pair_counts, 'Zone codes or sample counts changed from source representation.'

sample_total = int(sample['sample_trips'].sum())
expanded_total = expanded['expanded_trips'].sum()
benchmark_difference = expanded_total - 16723044
print({'sample_trips': sample_total, 'intrazonal_trips_in_full_observed_universe': 15016,
       'legacy_internal_intrazonals': 15013, 'additional_trips_vs_legacy_scope': 433,
       'expanded_total': str(expanded_total), 'benchmark_difference': str(benchmark_difference),
       'sample_od_pairs': len(sample), 'expanded_od_pairs': len(expanded)})
display(reconciliation)
display(quality)

## Interpretación

La matriz muestral incluye 100.174 viajes; la versión histórica limitada al área interna incluía 99.741. La diferencia de 433 se mantiene en la matriz actual con los códigos originales, sin etiquetarla mediante una unión geográfica inferida. El total expandido se contrasta descriptivamente con 16.723.044. No se publican matrices por modo/período ni medidas de incertidumbre por celda; esta salida no pretende reproducir exactamente los productos oficiales.